# Name That LLM: an error map of the 18 models – and what extra data can (and cannot) fix

Everyone is adding data to this competition. Before generating more, it helps to know **where a model actually fails**: which of the 18 classes are confused with which, whether the errors are *between* families (Claude vs GPT) or *inside* them (gpt-5.6-sol vs gpt-5.6-terra), and whether the weak classes are the ones that are short of data.

**What this notebook does (CPU only, ~25 min):**
1. trains the public 4-view n-gram `LinearSVC` on train + the public pool, with prompt-grouped 5-fold CV and a leakage guard;
2. draws the per-class recall and the confusion map, lists the most confused pairs and splits the error into *wrong family* vs *right family, wrong model*;
3. puts next to it how much public pool data each class already has, and which classes can be generated through the Kaggle Benchmarks model quota.

**Setup (fixed before looking at results):** `StratifiedGroupKFold(5, shuffle=True, random_state=42)` grouped by `prompt_id`; vocabularies fitted on train + pool only; pool passages enter training folds only, and any pool passage whose content TF-IDF is too close to a validation passage is dropped (threshold: 99th percentile of the nearest *other-prompt* similarity in train); metric – balanced accuracy.

Credits: public pool – Nishant Dahal and the 14 sources listed in that dataset; the 4-view n-gram model – yangkuangou; the leakage-guard idea follows the public two-stage LoRA notebooks.


In [ ]:
import re, time, gc, glob, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack, vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix
def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True)); assert hits, f'attach the dataset with {name}'; return hits[0]
tr = pd.read_csv(find('train.csv')); pool = pd.read_parquet(find('pool_v1.parquet'))
C = np.array(sorted(tr.model.unique())); K = len(C); y = np.searchsorted(C, tr.model.values)
pool = pool[pool.model.isin(C)].reset_index(drop=True); pool['text'] = pool.text.astype(str); yp = np.searchsorted(C, pool.model.values)
def family(m):
    for f in ('claude', 'gemini', 'gpt', 'glm', 'kimi', 'qwen'):
        if m.startswith(f): return f
FAM = np.array([family(c) for c in C])
SCALE = float(os.environ.get('NTL_SCALE', 1.0))   # 1.0 on Kaggle; smaller only for a quick local test
print('train', tr.shape, '| pool', pool.shape, '| classes', K, '| families', dict(pd.Series(FAM).value_counts()))

## 1. Out-of-fold predictions
One model, one set of folds; every number below comes from these out-of-fold scores.

In [ ]:
FW = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
shape = lambda t: re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)
func = lambda t: " ".join(w.lower() if w.lower() in FW or not w.isalnum() else "CONTENT" for w in re.findall(r"\w+|[^\w\s]", t))
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, .7), ("shape", 2, 6, 60000, .5), ("function", 1, 3, 50000, .5)]
texts = tr.text.tolist() + pool.text.tolist(); n0 = len(tr)
blocks = []
for view, lo, hi, mf, w in VIEWS:
    prep = shape if view == "shape" else func if view == "function" else (lambda s: s)
    an = "char" if view == "shape" else "word" if view == "function" else view
    o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=int(mf * SCALE), dtype=np.float32)
    if an == "word": o["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    blocks.append(TfidfVectorizer(**o).fit_transform([prep(t) for t in texts]) * w)
X = hstack(blocks, format='csr'); del blocks; gc.collect(); Xtr, Xpo = X[:n0], X[n0:]
vc = TfidfVectorizer(stop_words='english', sublinear_tf=True, min_df=2, dtype=np.float32).fit(texts)
XC = vc.transform(texts); XCtr, XCpo = XC[:n0], XC[n0:]
groups = pd.factorize(np.asarray(tr.prompt_id, dtype=object))[0]
S = (XCtr @ XCtr.T).toarray(); np.fill_diagonal(S, -1)
TH = float(np.clip(np.quantile(np.where(groups[:, None] != groups[None, :], S, -1).max(1), .99), .1, .5)); del S
print(f'leakage-guard threshold {TH:.3f}')
t0 = time.time(); oof = np.zeros((n0, K)); dropped = []
for tri, va in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(tr.text, y, groups):
    kp = np.asarray((XCpo @ XCtr[va].T).max(1).toarray()).ravel() < TH; dropped.append(int((~kp).sum()))
    m = LinearSVC(C=.5, class_weight='balanced', max_iter=5000, random_state=42).fit(vstack([Xtr[tri], Xpo[kp]]), np.r_[y[tri], yp[kp]])
    oof[va] = m.decision_function(Xtr[va])
pred = oof.argmax(1)
print(f'OOF balanced accuracy {balanced_accuracy_score(y, pred):.4f} | pool passages dropped by the guard per fold {dropped} | {time.time() - t0:.0f}s')

## 2. Per-class recall
Balanced accuracy is the mean of these 18 numbers, so every class weighs the same.

In [ ]:
rec = pd.Series(recall_score(y, pred, average=None), index=C)
cm = confusion_matrix(y, pred, labels=range(K))
off = cm.copy(); np.fill_diagonal(off, 0)
tab = pd.DataFrame({'family': FAM, 'recall': rec.values, 'most often predicted as': C[off.argmax(1)],
                    'share of its errors': off.max(1) / np.maximum(off.sum(1), 1), 'train passages': np.bincount(y, minlength=K)}, index=C)
display(tab.sort_values('recall').round(2))

## 3. The confusion map
Rows are the true model, columns the predicted one, each row normalised to 1. Thick lines separate families.

In [ ]:
order = np.argsort(FAM, kind='stable'); cmn = cm[order][:, order] / cm.sum(1, keepdims=True)[order]
fig, ax = plt.subplots(figsize=(11, 9)); im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=.8)
ax.set_xticks(range(K)); ax.set_xticklabels(C[order], rotation=90); ax.set_yticks(range(K)); ax.set_yticklabels(C[order])
for i in range(K):
    for j in range(K):
        if cmn[i, j] >= .08: ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if cmn[i, j] > .45 else 'black')
f = FAM[order]; cuts = [i - .5 for i in range(1, K) if f[i] != f[i - 1]]
for c in cuts: ax.axhline(c, color='k', lw=1.5); ax.axvline(c, color='k', lw=1.5)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); plt.colorbar(im, fraction=.04); plt.tight_layout(); plt.show()

## 4. The most confused pairs
Counted both ways (A predicted as B plus B predicted as A), out of about 300 passages per pair.

In [ ]:
pairs = [(C[i], C[j], int(cm[i, j] + cm[j, i]), FAM[i] == FAM[j]) for i in range(K) for j in range(i + 1, K)]
pairs = pd.DataFrame(pairs, columns=['model A', 'model B', 'confusions', 'same family']).sort_values('confusions', ascending=False)
display(pairs.head(12).reset_index(drop=True))
print(f"top-12 pairs: {pairs.head(12)['same family'].mean():.0%} are inside one family")

## 5. Wrong family, or right family but wrong model?
If most errors stay inside a family, the hard problem is telling sibling models apart, and family-level features will not help much.

In [ ]:
fam_true, fam_pred = FAM[y], FAM[pred]; wrong = pred != y
inside = (wrong & (fam_true == fam_pred)).sum(); between = (wrong & (fam_true != fam_pred)).sum()
print(f'errors: {wrong.sum()} | inside the family: {inside} ({inside / wrong.sum():.0%}) | wrong family: {between} ({between / wrong.sum():.0%})')
print(f'family accuracy: {(fam_true == fam_pred).mean():.3f}')
rows = []
for f in sorted(set(FAM)):
    m = fam_true == f; mf = m & (fam_pred == f)
    rows.append({'family': f, 'models': int((FAM == f).sum()), 'family recall': (fam_pred[m] == f).mean(),
                 'model accuracy when the family is right': (pred[mf] == y[mf]).mean() if (mf.any() and (FAM == f).sum() > 1) else np.nan})
display(pd.DataFrame(rows).set_index('family').round(3))

## 6. Is a weak class just short of data?

Two facts next to the recall:
- **pool passages** – how many public pool passages each class already has;
- **Kaggle Benchmarks** – whether the model is in the Benchmarks model list, i.e. whether its answers can be generated with the free model quota. The list was read on 29 Oct 2026 inside a Benchmarks task notebook with `os.environ['LLMS_AVAILABLE']`; it can change, so re-check it before planning.

Generating data is allowed only within the competition rules (section 6: external data must be publicly available or of "minimal cost" under the Reasonableness Standard) – check them and the Benchmarks terms yourself before you rely on this route.

In [ ]:
BENCHMARKS = {  # competition class -> name in the Benchmarks model list (29 Oct 2026); None = not listed
    'claude-fable-5': None, 'claude-opus-4-6': 'anthropic/claude-opus-4-6', 'claude-opus-4-8': 'anthropic/claude-opus-4-8',
    'claude-opus-5': 'anthropic/claude-opus-5', 'claude-sonnet-5': 'anthropic/claude-sonnet-5',
    'gemini-3.1-pro-preview': 'google/gemini-3.1-pro-preview', 'gemini-3.6-flash': 'google/gemini-3.6-flash',
    'gemini-3.7-flash': None, 'gemini-3.8-flash': 'google/gemini-3.8-flash', 'glm-5.3': None, 'glm-5.3-flash': None,
    'gpt-5.5': 'openai/gpt-5.5-2026-04-23', 'gpt-5.6-luna': 'openai/gpt-5.6-luna', 'gpt-5.6-sol': 'openai/gpt-5.6-sol',
    'gpt-5.6-terra': 'openai/gpt-5.6-terra', 'gpt-6-astra': 'openai/gpt-6-astra', 'kimi-k3': None, 'qwen3.8-max': None}
d = tab[['family', 'recall', 'most often predicted as']].copy()
d['pool passages'] = pool.model.value_counts().reindex(C).fillna(0).astype(int).values
d['in Benchmarks'] = ['yes' if BENCHMARKS[c] else 'no' for c in C]
display(d.sort_values('recall').round(2))
r = np.corrcoef(d['pool passages'], d.recall)[0, 1]
print(f'correlation of recall with pool passages across the 18 classes: {r:.2f}')
for g, s in d.groupby('in Benchmarks').recall: print(f'in Benchmarks = {g}: {len(s)} classes, mean recall {s.mean():.2f}')
fig, ax = plt.subplots(figsize=(7, 5))
for g, mk in (('yes', 'o'), ('no', 'x')):
    s = d[d['in Benchmarks'] == g]; ax.scatter(s['pool passages'], s.recall, marker=mk, label=f'in Benchmarks: {g}')
for c, row in d.iterrows(): ax.annotate(c, (row['pool passages'], row.recall), fontsize=7, xytext=(3, 2), textcoords='offset points')
ax.set_xlabel('public pool passages'); ax.set_ylabel('OOF recall'); ax.legend(); plt.tight_layout(); plt.show()

## 7. Takeaways

*All numbers are from the run above (OOF balanced accuracy 0.435 with this model and these folds).*

- **Errors come in two kinds, and they need different fixes.** Section 5 splits them almost exactly in half: about half are *right family, wrong model* (siblings such as two Gemini Flash versions or two GPT-5.6 variants), the other half are *wrong family* (families that write alike). The top pairs in section 4 show both kinds.
- **More passages is not more recall.** Every class already has 1,100–2,700 pool passages, and recall is not related to that count (correlation in section 6 is close to zero). Some classes with the largest pools are among the weakest: their bottleneck is their siblings, not the amount of data.
- **The weakest classes are the ones you cannot generate.** The six classes missing from the Benchmarks list have a mean recall of about 0.28, against about 0.51 for the twelve listed ones. Generating data only for the listed classes would pull predictions towards them, and balanced accuracy, which weighs every class equally, can drop. If you do it, keep the class balance or use the new data only in a separate model for one family.
- **A cheap test before spending:** train a family-only model on the data you already have and measure accuracy *inside that family* on prompt-grouped folds. If it does not beat the main model there, a few hundred extra passages are unlikely to change much.

Feedback and other model views of the same map (LoRA, embeddings) are welcome in the comments. 
